# Sparse Matrix Computation in C Using CSR Representation

## Study Manual / Jupyter Notebook

This notebook implements Sparse Matrix–Vector Multiplication (SpMV) using **CSR (Compressed Sparse Row)** representation.

Learning objectives:
- Understand sparse matrices and NNZ.
- Understand CSR representation.
- Understand `values[]`, `col_index[]`, and `row_ptr[]`.
- Implement CSR matrix-vector multiplication in C.
- Verify the result manually.


## 1. Example Sparse Matrix

```text
A =
[5  0  0  8]
[0  3  0  0]
[0  0  6  0]
[2  0  0  7]
```

There are 4 rows, 4 columns, and 6 non-zero elements.

```text
ROWS = 4
COLS = 4
NNZ  = 6
```


## 2. CSR Representation

**CSR = Compressed Sparse Row**.

CSR uses three arrays:

1. `values[]` — non-zero values
2. `col_index[]` — column index of each non-zero value
3. `row_ptr[]` — starting position of each row

For this matrix:

```text
values     = {5, 8, 3, 6, 2, 7}
col_index  = {0, 3, 1, 2, 0, 3}
row_ptr    = {0, 2, 3, 4, 6}
```


## 3. How `row_ptr[]` Works

`row_ptr[r]` and `row_ptr[r+1]` identify the section of `values[]` belonging to row `r`.

```text
row_ptr = {0, 2, 3, 4, 6}

Row 0 → values[0..1]
Row 1 → values[2..2]
Row 2 → values[3..3]
Row 3 → values[4..5]
```


In [ ]:
values = [5, 8, 3, 6, 2, 7]
col_index = [0, 3, 1, 2, 0, 3]
row_ptr = [0, 2, 3, 4, 6]
x = [1, 2, 3, 4]

print('values    =', values)
print('col_index =', col_index)
print('row_ptr   =', row_ptr)
print('X         =', x)


## 4. CSR Matrix–Vector Multiplication

We calculate `Y = A × X`.

Algorithm:

```text
for each row r:
    Y[r] = 0
    for j = row_ptr[r] to row_ptr[r+1]-1:
        Y[r] += values[j] * X[col_index[j]]
```


In [ ]:
%%writefile sparse_csr.c
#include <stdio.h>

#define ROWS 4
#define COLS 4
#define NNZ 6

void sparseMatVecCSR(int values[], int col_index[], int row_ptr[], int x[], int y[])
{
    for (int row = 0; row < ROWS; row++)
    {
        y[row] = 0;

        for (int j = row_ptr[row]; j < row_ptr[row + 1]; j++)
        {
            y[row] += values[j] * x[col_index[j]];
        }
    }
}

int main()
{
    int values[NNZ] = {5, 8, 3, 6, 2, 7};
    int col_index[NNZ] = {0, 3, 1, 2, 0, 3};
    int row_ptr[ROWS + 1] = {0, 2, 3, 4, 6};
    int x[COLS] = {1, 2, 3, 4};
    int y[ROWS] = {0};

    printf("=====================================\n");
    printf(" Sparse Matrix Using CSR in C\n");
    printf("=====================================\n\n");

    printf("CSR Values:\n");
    for (int i = 0; i < NNZ; i++) printf("%d ", values[i]);

    printf("\n\nColumn Indices:\n");
    for (int i = 0; i < NNZ; i++) printf("%d ", col_index[i]);

    printf("\n\nRow Pointer:\n");
    for (int i = 0; i < ROWS + 1; i++) printf("%d ", row_ptr[i]);

    printf("\n\nInput Vector:\n");
    for (int i = 0; i < COLS; i++) printf("%d ", x[i]);

    sparseMatVecCSR(values, col_index, row_ptr, x, y);

    printf("\n\nOutput Vector:\n");
    for (int i = 0; i < ROWS; i++)
        printf("Y[%d] = %d\n", i, y[i]);

    return 0;
}


In [ ]:
!gcc sparse_csr.c -o sparse_csr
!./sparse_csr


## 5. Expected Output

```text
CSR Values:
5 8 3 6 2 7

Column Indices:
0 3 1 2 0 3

Row Pointer:
0 2 3 4 6

Input Vector:
1 2 3 4

Output Vector:
Y[0] = 37
Y[1] = 6
Y[2] = 18
Y[3] = 30
```


## 6. Manual Verification

```text
Y[0] = 5×X[0] + 8×X[3] = 5×1 + 8×4 = 37
Y[1] = 3×X[1]                 = 3×2     = 6
Y[2] = 6×X[2]                 = 6×3     = 18
Y[3] = 2×X[0] + 7×X[3] = 2×1 + 7×4 = 30
```


## 7. COO vs CSR

| Feature | COO | CSR |
|---|---|---|
| Values | Yes | Yes |
| Column indices | Yes | Yes |
| Row information | `row[]` | `row_ptr[]` |
| Representation | Row + column + value | Value + column + row pointer |
| Row-based processing | Moderate | Excellent |

COO explicitly stores the row of every non-zero element. CSR compresses this information using `row_ptr[]`.


## 8. Exercises

1. Create a 5×5 sparse matrix and represent it using CSR.
2. Write a C program to convert a dense matrix to CSR.
3. Write a C program to convert CSR back to a dense matrix.
4. Implement CSR matrix-vector multiplication in CUDA.
5. Compare COO and CSR implementations.
